# Robot Cell Scheduling: Job-Shop Exercise

In this lab we schedule work inside a small automated production cell. The story is concrete, but the optimization problem is the classic **job-shop scheduling** problem.

The goal is to choose operation start times so that:

1. each kit follows its required route,
2. no station works on two kits at the same time,
3. all kits finish as early as possible.

The final completion time is called the **makespan** and is usually written as $C_{max}$.

## Learning goals

After this exercise you should be able to:

1. translate a production-cell story into jobs, stations, operations, and durations,
2. explain why binary variables are needed when two operations compete for the same station,
3. build a small mixed-integer linear programming model in PuLP,
4. validate a computed schedule before trusting the plot,
5. interpret a Gantt chart for a production schedule.


## Story: the last minutes before the demo

A robotics lab is preparing four demo kits before visitors arrive. Each kit must visit three automated stations, but the route is not the same for every kit.

The stations are:

- **Laser cutter**: cuts or trims parts.
- **Robot arm**: assembles parts.
- **Vision test**: checks the kit with a camera.

Only one kit can use a station at a time. A kit may wait between stations, but it cannot skip or reorder its own operations.


## Robot-cell picture

![Robot-cell scheduling illustration](job_illustration.png)

This picture shows the task as a shared robot cell: each kit has a fixed route, stations can process only one kit at a time, and waiting kits stay in the buffer area.


In [ ]:
# Install once if PuLP or the plotting/data libraries are not available:
# %pip install pulp pandas matplotlib

import pandas as pd

# Data for the main exercise.
# Each job is a route: (station, processing_time_in_minutes).
jobs = {
    "Frame kit": [
        ("Laser cutter", 3),
        ("Robot arm", 4),
        ("Vision test", 2),
    ],
    "Sensor mast": [
        ("Robot arm", 2),
        ("Laser cutter", 4),
        ("Vision test", 3),
    ],
    "Gripper jaw": [
        ("Laser cutter", 2),
        ("Vision test", 1),
        ("Robot arm", 3),
    ],
    "Battery tray": [
        ("Vision test", 2),
        ("Robot arm", 3),
        ("Laser cutter", 3),
    ],
}

operation_rows = []
for job, route in jobs.items():
    for operation, (station, duration) in enumerate(route, start=1):
        operation_rows.append({
            "job": job,
            "operation": operation,
            "station": station,
            "duration": duration,
        })

operations = pd.DataFrame(operation_rows)
operations


## Read the data as operations

Each row is one operation. For example, the **Frame kit** must first use the laser cutter for 3 minutes, then the robot arm for 4 minutes, then the vision test for 2 minutes.

This table is the input to the scheduling model. The solver will decide only the start times and the order of competing operations on each station.


## Route diagram

The table is useful for exact data. The route diagram below shows the same information as paths through the cell.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch


def plot_job_routes(jobs):
    station_colors = {
        "Laser cutter": "#F8D7DA",
        "Robot arm": "#DDEED8",
        "Vision test": "#D8E8F8",
    }

    job_names = list(jobs)
    max_operations = max(len(route) for route in jobs.values())
    fig, ax = plt.subplots(figsize=(10, 0.9 * len(job_names) + 1.5))
    ax.set_xlim(0.4, max_operations + 0.6)
    ax.set_ylim(-0.6, len(job_names) - 0.4)
    ax.axis("off")

    for y, job in enumerate(reversed(job_names)):
        ax.text(0.1, y, job, ha="right", va="center", fontsize=10, weight="bold")
        route = jobs[job]
        for operation, (station, duration) in enumerate(route, start=1):
            x = operation
            box = FancyBboxPatch(
                (x - 0.38, y - 0.25),
                0.76,
                0.5,
                boxstyle="round,pad=0.03",
                linewidth=1,
                edgecolor="#333333",
                facecolor=station_colors.get(station, "#EFEFEF"),
            )
            ax.add_patch(box)
            ax.text(x, y, f"{station}\n{duration} min", ha="center", va="center", fontsize=8)
            if operation < len(route):
                ax.annotate("", xy=(x + 0.53, y), xytext=(x + 0.38, y), arrowprops={"arrowstyle": "->", "lw": 1.1})

    for operation in range(1, max_operations + 1):
        ax.text(operation, len(job_names) - 0.15, f"Operation {operation}", ha="center", va="bottom", fontsize=9)

    ax.set_title("Required route for each kit", pad=12)
    return fig, ax


plot_job_routes(jobs);


## Warm-up: two jobs and two stations

Before building the general model, look at a tiny case by hand.

- Job A: `Station 1` for 5 minutes, then `Station 2` for 6 minutes.
- Job B: `Station 2` for 4 minutes, then `Station 1` for 7 minutes.

Questions:

1. Can both first operations start at time 0?
2. What must happen before Job A can use Station 2?
3. What must happen before Job B can use Station 1?
4. Draw a schedule manually. The optimal makespan is **12 minutes**.

The useful idea is this: on each station, either Job A goes before Job B or Job B goes before Job A. That either/or decision is what the binary variable represents.


In [ ]:
warmup_jobs = {
    "Job A": [
        ("Station 1", 5),
        ("Station 2", 6),
    ],
    "Job B": [
        ("Station 2", 4),
        ("Station 1", 7),
    ],
}

warmup_rows = []
for job, route in warmup_jobs.items():
    for operation, (station, duration) in enumerate(route, start=1):
        warmup_rows.append({
            "job": job,
            "operation": operation,
            "station": station,
            "duration": duration,
        })

pd.DataFrame(warmup_rows)


## Model idea

For every operation we create a start-time variable:

$$s_{j,k} \ge 0$$

where $j$ is the job and $k$ is the operation number inside that job.

The model has three groups of constraints.

### 1. Route order inside each job

If operation 2 of a job comes after operation 1, then operation 2 cannot start before operation 1 finishes:

$$s_{j,2} \ge s_{j,1} + p_{j,1}$$

### 2. No overlap on the same station

For two operations `a` and `b` that need the same station, exactly one must happen first. We use a binary variable $y_{a,b}$.

If $y_{a,b}=1$, operation `a` is before `b`. If $y_{a,b}=0$, operation `b` is before `a`.

The two Big-M constraints are:

$$s_b \ge s_a + p_a - M(1-y_{a,b})$$

$$s_a \ge s_b + p_b - My_{a,b}$$

The Big-M term is a switch. When one ordering is selected, one constraint is active and blocks overlap. The other constraint receives a large subtraction, so it becomes loose and does not restrict the schedule. $M$ must be large enough to relax the inactive constraint, but not absurdly large. Here we use the sum of all operation durations because that is a safe schedule horizon for this basic model: even a one-at-a-time schedule would finish by then. If you later add fixed release times, maintenance windows, or other constraints that can force idle time, increase the horizon so Big-M is still large enough.

### 3. Makespan

The makespan must be at least the finish time of every job's last operation:

$$C_{max} \ge s_{j,last} + p_{j,last}$$

The objective is:

$$\min C_{max}$$


In [ ]:
from itertools import combinations
import re

import pulp


def _safe_name(text):
    """Create solver-friendly variable and constraint names."""
    return re.sub(r"[^A-Za-z0-9_]+", "_", str(text)).strip("_")


def _operation_name(operation):
    job, index = operation
    return f"{_safe_name(job)}_op{index + 1}"


def solve_jobshop(jobs, msg=False, horizon=None):
    """Solve a small job-shop scheduling instance.

    Parameters
    ----------
    jobs : dict[str, list[tuple[str, int | float]]]
        Mapping from job name to an ordered route. Each route item is
        (station_name, processing_time).
    msg : bool
        Whether the solver should print its log.
    horizon : int | float | None
        Big-M value. If None, use the sum of all processing times.

    Returns
    -------
    makespan : float
    schedule : pandas.DataFrame
        Columns: job, operation, station, start, duration, finish.
    model : pulp.LpProblem
        The solved PuLP model, useful for inspection.
    """
    model = pulp.LpProblem("Robot_Cell_Job_Shop", pulp.LpMinimize)

    operations = [
        (job, operation_index)
        for job, route in jobs.items()
        for operation_index in range(len(route))
    ]
    station_of = {
        (job, operation_index): jobs[job][operation_index][0]
        for job, operation_index in operations
    }
    duration = {
        (job, operation_index): jobs[job][operation_index][1]
        for job, operation_index in operations
    }
    if horizon is None:
        horizon = sum(duration[operation] for operation in operations)

    start = {
        operation: pulp.LpVariable(
            f"start__{_operation_name(operation)}",
            lowBound=0,
            cat="Continuous",
        )
        for operation in operations
    }
    cmax = pulp.LpVariable("Cmax", lowBound=0, cat="Continuous")

    model += cmax, "minimize_makespan"

    # Route-order constraints: operations inside a job must follow the route.
    for job, route in jobs.items():
        for operation_index in range(len(route) - 1):
            current_operation = (job, operation_index)
            next_operation = (job, operation_index + 1)
            model += (
                start[next_operation] >= start[current_operation] + duration[current_operation],
                f"route__{_operation_name(current_operation)}__before__{_operation_name(next_operation)}",
            )

    # No-overlap constraints: two operations sharing a station cannot overlap.
    stations = sorted({station for route in jobs.values() for station, _ in route})
    for station in stations:
        station_operations = [operation for operation in operations if station_of[operation] == station]
        for first_operation, second_operation in combinations(station_operations, 2):
            order = pulp.LpVariable(
                f"order__{_safe_name(station)}__{_operation_name(first_operation)}__before__{_operation_name(second_operation)}",
                cat="Binary",
            )
            model += (
                start[second_operation]
                >= start[first_operation] + duration[first_operation] - horizon * (1 - order),
                f"no_overlap__{_safe_name(station)}__{_operation_name(first_operation)}__before__{_operation_name(second_operation)}",
            )
            model += (
                start[first_operation]
                >= start[second_operation] + duration[second_operation] - horizon * order,
                f"no_overlap__{_safe_name(station)}__{_operation_name(second_operation)}__before__{_operation_name(first_operation)}",
            )

    # Makespan constraints: Cmax must be after the final operation of every job.
    for job, route in jobs.items():
        last_operation = (job, len(route) - 1)
        model += (
            cmax >= start[last_operation] + duration[last_operation],
            f"makespan_after__{_safe_name(job)}",
        )

    solver = pulp.PULP_CBC_CMD(msg=msg)
    model.solve(solver)

    status = pulp.LpStatus[model.status]
    if status != "Optimal":
        raise RuntimeError(
            f"Solver status is {status}, not Optimal. "
            "Check whether the routes and no-overlap constraints are feasible. "
            "If you added release times or maintenance windows, also check that the horizon/Big-M value is large enough."
        )

    rows = []
    for job, route in jobs.items():
        for operation_index, (station, processing_time) in enumerate(route):
            operation = (job, operation_index)
            start_time = pulp.value(start[operation])
            if start_time is None:
                raise RuntimeError(
                    f"No start time was returned for {job}, operation {operation_index + 1}. "
                    "Check the solver status and model constraints."
                )
            rows.append({
                "job": job,
                "operation": operation_index + 1,
                "station": station,
                "start": start_time,
                "duration": processing_time,
                "finish": start_time + processing_time,
            })

    schedule = (
        pd.DataFrame(rows)
        .sort_values(["station", "start", "job"])
        .reset_index(drop=True)
    )
    return pulp.value(cmax), schedule, model


In [ ]:
warmup_makespan, warmup_schedule, warmup_model = solve_jobshop(warmup_jobs)
print(f"Warm-up optimal makespan: {warmup_makespan:.0f} minutes")
print("Expected value for the original warm-up data: 12 minutes")
warmup_schedule


In [ ]:
makespan, schedule, model = solve_jobshop(jobs)
print(f"Robot-cell optimal makespan: {makespan:.0f} minutes")
print("Expected value for the original robot-cell data: 13 minutes")
schedule


## Validate before plotting

A Gantt chart helps us see the schedule, but first check the rules automatically.

A valid schedule must satisfy two things:

1. every job follows its operation route,
2. operations on the same station do not overlap.


In [ ]:
def validate_schedule(jobs, schedule, tolerance=1e-7):
    """Check route-order and no-overlap rules for a computed schedule."""
    indexed = {
        (row.job, int(row.operation) - 1): row
        for row in schedule.itertuples(index=False)
    }

    for job, route in jobs.items():
        for operation_index in range(len(route) - 1):
            current_operation = indexed[(job, operation_index)]
            next_operation = indexed[(job, operation_index + 1)]
            if next_operation.start + tolerance < current_operation.finish:
                raise AssertionError(
                    f"Route violation for {job}: operation {operation_index + 2} "
                    f"starts before operation {operation_index + 1} finishes."
                )

    for station, station_schedule in schedule.groupby("station"):
        ordered = station_schedule.sort_values("start")
        previous_operation = None
        for operation in ordered.itertuples(index=False):
            if (
                previous_operation is not None
                and operation.start + tolerance < previous_operation.finish
            ):
                raise AssertionError(
                    f"Overlap on {station}: {previous_operation.job} and {operation.job}."
                )
            previous_operation = operation

    return "Schedule is valid."


validate_schedule(warmup_jobs, warmup_schedule), validate_schedule(jobs, schedule)


In [ ]:
import matplotlib.pyplot as plt


def plot_gantt(schedule, title="Optimized robot-cell schedule"):
    """Plot a Gantt chart from a schedule DataFrame."""
    stations = list(schedule["station"].drop_duplicates())
    jobs_in_schedule = list(schedule["job"].drop_duplicates())
    y_position = {station: index for index, station in enumerate(stations)}
    color_map = {
        job: plt.get_cmap("tab10")(index % 10)
        for index, job in enumerate(jobs_in_schedule)
    }

    fig_height = max(3.5, 0.8 * len(stations) + 1.5)
    fig, ax = plt.subplots(figsize=(10, fig_height))
    labelled_jobs = set()

    for row in schedule.itertuples(index=False):
        y = y_position[row.station]
        label = row.job if row.job not in labelled_jobs else None
        labelled_jobs.add(row.job)

        ax.barh(
            y,
            row.duration,
            left=row.start,
            color=color_map[row.job],
            edgecolor="black",
            label=label,
        )
        operation_label = f"op {int(row.operation)}" if row.duration >= 1.8 else str(int(row.operation))
        ax.text(
            row.start + row.duration / 2,
            y,
            operation_label,
            ha="center",
            va="center",
            fontsize=8,
        )

    ax.set_yticks(list(y_position.values()))
    ax.set_yticklabels(list(y_position.keys()))
    ax.set_xlabel("Time [minutes]")
    ax.set_ylabel("Station")
    ax.set_title(title)
    ax.set_xlim(0, schedule["finish"].max() + 1)
    ax.grid(axis="x", linestyle=":")
    ax.set_axisbelow(True)
    ax.legend(title="Job", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    return fig, ax


plot_gantt(schedule);


## Student tasks

### Task 1: identify the optimization objects

Fill in the blanks:

- Jobs are: `__________`
- Stations are: `__________`
- Decision variables are: `__________`
- The objective is: `__________`
- The two main constraint families are: `__________`

### Task 2: modify the data

The robot arm gets a faster gripper, so every **Robot arm** operation is 1 minute shorter, but never below 1 minute. Re-solve the model. How much does the makespan improve?

### Task 3: bottleneck analysis

Using the Gantt chart, answer:

- Which station is busiest?
- Which station has the most idle time?
- Which job finishes last?
- Is the last-finishing job necessarily the bottleneck? Explain.

### Task 4: add a maintenance window

The vision test station is unavailable from time 4 to time 6. Extend the model so no vision-test operation can overlap that interval.

Hint: this is another either/or decision. For each vision-test operation, it must finish before minute 4 or start after minute 6.

### Challenge: due dates

Suppose visitors care most about the `Frame kit`, which has a due date at minute 10. Add a tardiness variable.

A completion time $C_j$ tells us when job $j$ finishes. A due date $d_j$ tells us when it should finish. Tardiness is the delay after the due date:

$$T_j \ge C_j - d_j$$

Also add $T_j \ge 0$, because finishing early should give zero tardiness, not a negative penalty. For the `Frame kit`, if it finishes at minute 13 and the due date is minute 10, then $T_j=3$.

Then solve either:

- minimize makespan first, then tardiness,
- or minimize `makespan + 0.5 * total_tardiness`.
